<a href="https://colab.research.google.com/github/jxddavis-ui/mlsn_team_7_pnemouniaMNIST/blob/main/final_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
%%bash
pip install medmnist

In [8]:
import medmnist
from medmnist import INFO

# Get info about PneumoniaMNIST
data_flag = 'pneumoniamnist'
info = INFO[data_flag]
print("Dataset description:")
for key, val in info.items():
    if key != 'label':
        print(f"  {key}: {val}")
    else:
        print(f"  {key}: {list(val.items())}")

Dataset description:
  python_class: PneumoniaMNIST
  description: The PneumoniaMNIST is based on a prior dataset of 5,856 pediatric chest X-Ray images. The task is binary-class classification of pneumonia against normal. We split the source training set with a ratio of 9:1 into training and validation set and use its source validation set as the test set. The source images are gray-scale, and their sizes are (384−2,916)×(127−2,713). We center-crop the images and resize them into 1×28×28.
  url: https://zenodo.org/records/10519652/files/pneumoniamnist.npz?download=1
  MD5: 28209eda62fecd6e6a2d98b1501bb15f
  url_64: https://zenodo.org/records/10519652/files/pneumoniamnist_64.npz?download=1
  MD5_64: 8f4eceb4ccffa70c672198ea285246c6
  url_128: https://zenodo.org/records/10519652/files/pneumoniamnist_128.npz?download=1
  MD5_128: 05b46931834c231683c68f40c47b2971
  url_224: https://zenodo.org/records/10519652/files/pneumoniamnist_224.npz?download=1
  MD5_224: d6a3c71de1b945ea11211b03746c1f

In [9]:
from medmnist import PneumoniaMNIST
import numpy as np

# Load the training split of the dataset (downloads automatically to ~/.medmnist if not cached)
train_dataset = PneumoniaMNIST(split='train', download=True)

# Inspect data arrays directly
print("\n--- Data Inspection ---")
print(f"Images shape: {train_dataset.imgs.shape} (Number of samples, height, width)")
print(f"Labels shape: {train_dataset.labels.shape}")

# Check class balance
labels = train_dataset.labels.flatten()
unique, counts = np.unique(labels, return_counts=True)
for label, count in zip(unique, counts):
    label_name = info['label'][str(label)]
    print(f"Class {label} ({label_name}): {count} images ({count / len(labels):.1%})")

100%|██████████| 4.17M/4.17M [00:06<00:00, 622kB/s]



--- Data Inspection ---
Images shape: (4708, 28, 28) (Number of samples, height, width)
Labels shape: (4708, 1)
Class 0 (normal): 1214 images (25.8%)
Class 1 (pneumonia): 3494 images (74.2%)


In [11]:
%%writefile app.py
"""
Pediatric pneumonia X-ray demo (educational only).
Run with:  streamlit run app.py

Frozen settings come from the team's final evaluation (MLSN Team 7).
Items marked TODO need confirming with Nico.
"""
import os
import pandas as pd
import streamlit as st
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

# ---------------- FROZEN SETTINGS ----------------
WEIGHTS_PATH = "weights/best_model.pt"   # TODO: final weights file (LR 3e-5, seed 42, 15 epochs)
MODEL_NAME = "DenseNet121"
IMG_SIZE = 64
LEARNING_RATE = 3e-5
SEED = 42
EPOCHS = 15
CLASS_WEIGHTS = "None (unweighted)"
TRAIN_AUGMENTATION = "TODO: e.g. random flip, small rotation"   # fill in from the training notebook
THRESHOLD = 0.5                          # TODO: confirm the frozen threshold used for the test run
MEAN, STD = [0.5, 0.5, 0.5], [0.5, 0.5, 0.5]   # TODO: must match training normalization
SAMPLES_DIR = "samples"
# -------------------------------------------------

# ---------------- RESULTS (from the single test run) ----------------
RESULTS = pd.DataFrame(
    {
        "Recall": [0.9871, 0.9949],
        "Specificity": [0.9630, 0.6624],
        "AUC": [0.9981, 0.9830],
        "Accuracy": [0.9809, 0.8702],
    },
    index=["Validation", "Test"],
)
TP, TN, FP, FN = 388, 155, 79, 2
# ---------------------------------------------------------------------

DISCLAIMER = (
    "Educational demo, not for clinical use. "
    "Trained on pediatric chest X-rays (PneumoniaMNIST) from a single source."
)


def build_model():
    """Must match the training architecture exactly."""
    model = models.densenet121(weights=None)
    model.classifier = nn.Linear(model.classifier.in_features, 1)  # 1 logit
    return model


@st.cache_resource
def load_model():
    model = build_model()
    state = torch.load(WEIGHTS_PATH, map_location="cpu")
    model.load_state_dict(state)
    model.eval()
    return model


# Same as evaluation: grayscale -> resize -> 3 channels -> normalize. No augmentation.
preprocess = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


def predict(model, image):
    x = preprocess(image).unsqueeze(0)
    with torch.no_grad():
        prob = torch.sigmoid(model(x)).item()   # P(pneumonia)
    return ("Pneumonia" if prob >= THRESHOLD else "Normal"), prob


def true_label_from_name(name):
    n = name.lower()
    if "pneu" in n:
        return "Pneumonia"
    if "normal" in n:
        return "Normal"
    return None


# ------------------------------- UI ---------------------------------
st.set_page_config(page_title="Pediatric Pneumonia Demo", page_icon="🫁", layout="centered")
st.title("Pediatric Chest X-ray: Pneumonia Detector")
st.warning(DISCLAIMER)

tab_predict, tab_results, tab_about = st.tabs(["Try the model", "Test results", "Settings & limitations"])

# ---------------- Tab 1: predict ----------------
with tab_predict:
    model = load_model()
    image, true_label = None, None

    source = st.radio("Choose an input", ["Sample image", "Upload your own"], horizontal=True)
    if source == "Upload your own":
        f = st.file_uploader("Upload a chest X-ray", type=["png", "jpg", "jpeg"])
        if f:
            image = Image.open(f).convert("L")
    else:
        if os.path.isdir(SAMPLES_DIR):
            files = sorted(x for x in os.listdir(SAMPLES_DIR) if x.lower().endswith((".png", ".jpg", ".jpeg")))
            choice = st.selectbox("Pick a sample test image", files)
            if choice:
                image = Image.open(os.path.join(SAMPLES_DIR, choice)).convert("L")
                true_label = true_label_from_name(choice)
        else:
            st.info(f"Add test images to the '{SAMPLES_DIR}' folder.")

    if image is not None:
        label, prob = predict(model, image)
        c1, c2 = st.columns(2)
        with c1:
            st.image(image, caption="Input image", use_container_width=True)
        with c2:
            st.metric("Prediction", label)
            st.metric("P(pneumonia)", f"{prob:.1%}")
            st.caption(f"Decision threshold: {THRESHOLD}")
            if true_label:
                ok = "✅ correct" if true_label == label else "❌ incorrect"
                st.write(f"True label: **{true_label}** ({ok})")

        st.info(
            "Context: on the held-out test set this model caught almost all pneumonia cases "
            "(recall 99.5%) but wrongly flagged about a third of normal X-rays "
            "(specificity 66.2%). A 'Pneumonia' result is more likely to be a false alarm than a "
            "'Normal' result is to be a miss."
        )

        # --- Optional Grad-CAM (when Yuki's function is ready) ---
        # from gradcam import make_overlay
        # if st.checkbox("Show Grad-CAM overlay"):
        #     st.image(make_overlay(model, image), caption="Grad-CAM")

# ---------------- Tab 2: results ----------------
with tab_results:
    st.subheader("Validation vs. test (test set run once)")
    st.dataframe(RESULTS.style.format("{:.4f}"), use_container_width=True)

    st.subheader("Test confusion matrix")
    cm = pd.DataFrame(
        [[TN, FP], [FN, TP]],
        index=["Actual: Normal", "Actual: Pneumonia"],
        columns=["Predicted: Normal", "Predicted: Pneumonia"],
    )
    st.table(cm)
    st.caption(f"TP {TP} | TN {TN} | FP {FP} | FN {FN}  (n = {TP + TN + FP + FN})")

    st.subheader("Baselines")
    n_test, n_pneu = TP + TN + FP + FN, TP + FN
    base = n_pneu / n_test
    st.write(
        f"The test set is {n_pneu}/{n_test} = **{base:.1%} pneumonia** (about 74% in train/val). "
        f"Always predicting 'pneumonia' would score **{base:.1%} accuracy** with 0% specificity, "
        "so accuracy alone is misleading here. Recall, specificity and AUC give the fuller picture."
    )

    st.subheader("Why is test worse than validation?")
    st.markdown(
        "- Specificity dropped from 96.3% to 66.2%, while recall stayed high and AUC stayed strong (0.983).\n"
        "- A high AUC with low specificity means the model **ranks cases well**, but the "
        "**decision threshold** is not well suited to the test distribution.\n"
        "- The model is trained **unweighted** on data that is about 74% pneumonia, so it leans toward "
        "predicting pneumonia. The test set has more normal cases (37.5%), which exposes this.\n"
        "- Results are reported as-is: the test set was run once and not used for retuning."
    )

# ---------------- Tab 3: settings & limitations ----------------
with tab_about:
    st.subheader("Frozen settings")
    st.table(pd.DataFrame({
        "Setting": ["Model", "Image size", "Class weights", "Learning rate", "Seed", "Epochs",
                    "Threshold", "Preprocessing", "Augmentation (training)", "Augmentation (at inference)"],
        "Value": [MODEL_NAME, f"{IMG_SIZE}x{IMG_SIZE}", CLASS_WEIGHTS, f"{LEARNING_RATE:g}", SEED,
                  f"{EPOCHS} (best validation AUC 0.9985)", THRESHOLD,
                  "Grayscale -> resize -> 3 channels -> normalize", TRAIN_AUGMENTATION, "None"],
    }))
    st.caption("Learning rate: 7 values tested from 1e-5 to 1e-2; 3e-5 gave the best validation results.")

    st.subheader("Limitations")
    st.markdown(
        "- Pediatric patients only, from a single source\n"
        "- Small, low-resolution images (64x64)\n"
        "- Many false positives on normal X-rays (specificity 66.2% on test)\n"
        "- Not validated for clinical use; educational demo only"
    )

st.divider()
st.caption(DISCLAIMER)

Overwriting app.py
